# C4 cross-encoder reranker: train on Colab

Before you run: **Runtime > Change runtime type > T4 GPU**. Upload `reranker_colab.zip` to the Colab file panel (left side), or put it in your Google Drive.

The zip is not kept in the repository. Create it on your computer with:

```
python -m agents.traceability_agent.reranker.data pairs
python -m agents.traceability_agent.reranker.data bundle
```

(this writes `data/reranker_colab.zip`; the pair files are regenerated identically).

Train = iTrust + SMOS + eANCI + eTour; test = LibEST + CM-1 + EBT (never seen in training).

In [ ]:
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: change the runtime type')

In [ ]:
!pip -q install transformers scikit-learn

In [ ]:
# 1) unpack the bundle (uploaded to /content). If it is on Drive: from google.colab import drive; drive.mount('/content/drive'); then use the Drive path.
!unzip -o -q /content/reranker_colab.zip -d /content/c4
%cd /content/c4
!ls data/reranker

## 1. Stage-1 baseline (no training; this is what you have to beat)

In [ ]:
!python evaluate.py --data data/reranker

## 2. CodeBERT (about 10-20 min on a T4)

In [ ]:
!python train.py train --data data/reranker --base microsoft/codebert-base --out model_codebert --epochs 3 --batch 16
!python train.py predict --data data/reranker --model model_codebert
!python evaluate.py --data data/reranker --scores model_codebert/test_scores.jsonl

## 3. MiniLM cross-encoder (the comparison)

In [ ]:
!python train.py train --data data/reranker --base cross-encoder/ms-marco-MiniLM-L-6-v2 --out model_minilm --epochs 3 --batch 16
!python train.py predict --data data/reranker --model model_minilm
!python evaluate.py --data data/reranker --scores model_minilm/test_scores.jsonl

## 4. Download the results
The zip contains `evaluation.json`, `training.json` and **`test_scores.jsonl`** of both models. Keep it: for the viva, copy the two `test_scores.jsonl` files to `project/data/reranker/colab_scores/` (as `codebert.jsonl` and `minilm.jsonl`) and the reproduction notebook re-evaluates them exactly. The model folders are large; keep them on Drive.

In [ ]:
!zip -q -r results.zip model_codebert/evaluation.json model_codebert/training.json model_codebert/test_scores.jsonl model_minilm/evaluation.json model_minilm/training.json model_minilm/test_scores.jsonl data/reranker/evaluation_stage1.json
from google.colab import files
files.download('results.zip')